# 🔄 Kaggle Git Sync

Ноутбук синхронизирует Git-репозиторий в Kaggle-окружении.

**Рабочий процесс:**
1. Локально редактируем код → `git commit` → `git push`
2. На Kaggle: **Run All** в этом ноутбуке → репозиторий обновлён → переходим в основной ноутбук и обучаем модель на GPU/TPU

---

## Настройка для приватного репозитория (один раз)

Если репозиторий **приватный**, добавьте GitHub Personal Access Token в Kaggle Secrets:

1. [GitHub](https://github.com) → **Settings** → **Developer settings** → **Personal access tokens** → **Generate new token**
   - Scope: `repo` (или `contents:read` для fine-grained token)
2. [Kaggle](https://kaggle.com) → аватар → **Settings** → **Secrets** → **Add new secret**
   - Name: `GITHUB_TOKEN`
   - Value: скопированный токен
3. В самом ноутбуке (Kaggle) → иконка 🔑 справа → включить `GITHUB_TOKEN`

Для **публичного** репозитория ничего настраивать не нужно.

In [8]:
# ============================================================
# КОНФИГУРАЦИЯ — меняйте только этот блок
# ============================================================

REPO_URL  = "https://github.com/your-username/your-repo"  # URL репозитория (без .git)
BRANCH    = "master"                                        # ветка
CLONE_DIR = "/kaggle/working/repo"                         # папка назначения в Kaggle

# ============================================================

In [9]:
# Получение GitHub токена из Kaggle Secrets
# Если токен не добавлен — работает только с публичными репозиториями

github_token = None
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("✓ GitHub токен найден — поддерживаются приватные репозитории")
except Exception:
    print("ℹ  GITHUB_TOKEN не найден — работаем в режиме публичного доступа")

ℹ  GITHUB_TOKEN не найден — работаем в режиме публичного доступа


In [10]:
import os
import subprocess

def run(cmd):
    """Выполняет shell-команду, выбрасывает исключение при ошибке."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Команда завершилась с ошибкой:\n{result.stderr.strip()}")
    return result.stdout.strip()

# Формируем URL: встраиваем токен если он есть
if github_token:
    proto, rest = REPO_URL.split("://", 1)
    auth_url = f"{proto}://{github_token}@{rest}.git"
else:
    auth_url = REPO_URL + ".git"

# Клонируем или обновляем
git_dir = os.path.join(CLONE_DIR, ".git")

if os.path.isdir(git_dir):
    print(f"Репозиторий существует в {CLONE_DIR} — обновляем...")
    run(f"git -C {CLONE_DIR} fetch origin")
    run(f"git -C {CLONE_DIR} reset --hard origin/{BRANCH}")
    run(f"git -C {CLONE_DIR} clean -fd")
    print("✓ Репозиторий обновлён до последнего коммита")
else:
    print(f"Клонируем репозиторий в {CLONE_DIR}...")
    os.makedirs(CLONE_DIR, exist_ok=True)
    run(f"git clone --branch {BRANCH} --depth 1 {auth_url} {CLONE_DIR}")
    print("✓ Репозиторий успешно склонирован")

Клонируем репозиторий в /kaggle/working/repo...


RuntimeError: Команда завершилась с ошибкой:
fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.

In [15]:
# Верификация: показываем последние коммиты и содержимое папки

print("=== Последние коммиты ===")
print(run(f"git -C {CLONE_DIR} log --oneline -5"))

print("\n=== Содержимое репозитория ===")
print(run(f"ls -la {CLONE_DIR}"))

=== Последние коммиты ===


RuntimeError: Команда завершилась с ошибкой:
fatal: not a git repository (or any of the parent directories): .git

In [ ]:
# (Опционально) Установка зависимостей из requirements.txt

req_path = os.path.join(CLONE_DIR, "requirements.txt")
if os.path.exists(req_path):
    print("Устанавливаем зависимости из requirements.txt...")
    run(f"pip install -q -r {req_path}")
    print("✓ Зависимости установлены")
else:
    print("requirements.txt не найден — пропускаем установку зависимостей")

## Готово!

Репозиторий находится в `CLONE_DIR`. Чтобы импортировать модули из него, добавьте в начало вашего основного ноутбука:

```python
import sys
sys.path.insert(0, "/kaggle/working/repo")  # или ваш CLONE_DIR
```

---

**Типичные проблемы:**

| Ошибка | Решение |
|--------|---------|
| `Authentication failed` | Добавьте `GITHUB_TOKEN` в Kaggle Secrets и включите его в ноутбуке |
| `Repository not found` | Проверьте `REPO_URL` (без `.git` в конце) |
| `fatal: not a git repository` | Удалите папку `CLONE_DIR` и запустите снова |